# Lecture 9: W&M HPC, Slurm Jobs, and Parallel Bootstrap Fits

**Unit 2, September 24, 2026**

Lecture 8 used a CMS Open Data dimuon sample to fit the $J/\psi\to\mu^+\mu^-$ mass peak and estimate uncertainties with bootstrap resampling. Today we use the same example to ask a practical computing question:

> What changes when the same calculation is too expensive, or too repetitive, to run interactively in one notebook?

The answer is not a new statistical method. It is a different workflow: put the repeated calculation in a command-line Python script, submit independent runs to the W&M HPC cluster with Slurm, and use the notebook to inspect and combine the finished CSV outputs.

## Learning Goals

By the end of this lecture you should be able to:

- connect to the W&M HPC login host `bora.sciclone.wm.edu` with `ssh`,
- create and activate a Python virtual environment on the cluster,
- explain why login nodes are for setup, editing, and short tests, not long calculations,
- run the Lecture 8 dimuon mass fit inside a command-line script,
- submit independent bootstrap jobs for different dimuon $p_T$ bins with Slurm,
- check job status with `squeue` and inspect output files after jobs finish,
- collect per-bin CSV outputs and plot the fitted results in this notebook.

The main cluster reference for this course is [`resources/wmhpc.md`](../resources/wmhpc.md). Keep that page open while working today.

# Part 1: Connecting to the W&M HPC Cluster

The practical target for today is the SSH host

```bash
ssh your_wm_username@bora.sciclone.wm.edu
```

Replace `your_wm_username` with your W&M username. If your local username already matches your W&M username, this shorter command may also work:

```bash
ssh bora.sciclone.wm.edu
```

Use a Bash-style terminal: macOS Terminal on macOS, or Git Bash inside VS Code on Windows. The page [`resources/vscode.md`](../resources/vscode.md) has the reminder for opening an integrated terminal and choosing Git Bash on Windows.

After logging in, you should be on a login node. Login nodes are shared entry points. Use them for short commands such as `pwd`, `ls`, `git status`, editing files, creating an environment, and submitting jobs. Do not run a long bootstrap loop directly on the login node.

### In-Class Setup Check, 15-20 minutes

Work through these steps with a neighbor. This part is intentionally interactive because SSH setup is the sort of thing where small local differences matter.

```bash
ssh your_wm_username@bora.sciclone.wm.edu
hostname
pwd
ls
```

If `ssh` asks whether you trust the host key, read the prompt, then answer `yes` if the host is `bora.sciclone.wm.edu`. If the login fails, check that you have requested HPC access and that your W&M password or multifactor authentication step completed successfully.

Once connected, move to the directory where you keep course work. If you cloned this repository on the cluster already, enter it. If not, use your normal course Git workflow to get a copy onto the cluster.

To clone the course repository through VS Code:

1. Open VS Code.
2. Open the Command Palette with `Shift` + `Command` + `P` on macOS or `Ctrl` + `Shift` + `P` on Windows.
3. Search for `Git: Clone` and select it.
4. Paste this repository URL:

```text
https://github.com/jrstevenjlab/gradcompphys.git
```

5. Choose the folder where you want to keep your course work.
6. When VS Code asks whether to open the cloned repository, choose `Open`.

You can also clone from a terminal with:

```bash
git clone https://github.com/jrstevenjlab/gradcompphys.git
cd gradcompphys
```

# Part 2: Python Environment on the Cluster

Use the same virtual-environment idea from [`resources/vscode.md`](../resources/vscode.md), but create the environment on the cluster inside this repository. On SciClone, first initialize the cluster shell environment so Slurm commands and environment modules are available. W&M describes this system on its [Environment Modules](https://www.wm.edu/offices/it/services/researchcomputing/using/modules/) page.

```bash
cd path/to/gradcompphys
source /usr/local/etc/sciclone.bashrc
which sbatch
module list
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install numpy scipy matplotlib pandas ipykernel ipython
```

If your account uses a `tcsh`/`csh` shell instead of Bash, use `source /usr/local/etc/sciclone.cshrc` for the SciClone environment setup. The rest of this lecture uses Bash syntax.

Check the active Python:

```bash
which python
which sbatch
python --version
python -m pip list
```

The SciClone environment makes cluster commands such as `sbatch`, `squeue`, and `module` available. The `.venv` controls Python packages for this repository. When you submit a Slurm job, the job starts in a fresh shell, so the batch job also needs to source the SciClone environment and activate `.venv` before running Python.

# Part 3: Local Recap of the Lecture 8 Calculation

Before using Slurm, always run a small version locally or interactively. Here the notebook repeats the basic Lecture 8 calculation: load the reduced CMS Open Data dimuon CSV, inspect mass and $p_T$, and fit the mass distribution with the same signal-plus-background likelihood model. The reusable fit code lives in [`scripts/lecture09/dimuon_pt_bootstrap.py`](../scripts/lecture09/dimuon_pt_bootstrap.py).

In [1]:
%matplotlib inline

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SCRIPT_DIR_CANDIDATES = [Path("../scripts/lecture09"), Path("scripts/lecture09")]
SCRIPT_DIR = next((path for path in SCRIPT_DIR_CANDIDATES if path.exists()), SCRIPT_DIR_CANDIDATES[0])
sys.path.insert(0, str(SCRIPT_DIR.resolve()))

from dimuon_pt_bootstrap import fit_mass_model, unpack_parameters

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

print(f"NumPy {np.__version__} | pandas {pd.__version__}")

NumPy 2.0.2 | pandas 2.3.3


In [2]:
DATA_PATH_CANDIDATES = [
    Path("../data/cms_dimuon_jpsi_3000.csv"),
    Path("data/cms_dimuon_jpsi_3000.csv"),
]
DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), DATA_PATH_CANDIDATES[0])
cms_data = pd.read_csv(DATA_PATH)
cms_data.head()

,Run,Event,dimuon_mass_GeV,dimuon_pt_GeV,leading_muon_pt_GeV,subleading_muon_pt_GeV,Q1,Q2,Type1,Type2,opposite_sign,both_global
0,146428,16748390,2.61363,6.645782,6.01508,1.072930,1,-1,G,T,True,False
1,146430,4871608,3.01719,12.358714,11.98250,0.715687,1,-1,G,T,True,False
2,146430,5571993,3.25675,9.647192,8.44892,1.691420,-1,1,G,T,True,False
3,146430,6127418,3.59661,12.836562,12.45830,0.865104,1,-1,G,T,True,False
4,146430,6371947,3.55104,10.580960,9.03751,2.105030,1,-1,G,T,True,False


In [3]:
summary = pd.DataFrame([
    {
        "events": len(cms_data),
        "mass_min_GeV": cms_data["dimuon_mass_GeV"].min(),
        "mass_max_GeV": cms_data["dimuon_mass_GeV"].max(),
        "pt_min_GeV": cms_data["dimuon_pt_GeV"].min(),
        "pt_max_GeV": cms_data["dimuon_pt_GeV"].max(),
    }
])
summary

,events,mass_min_GeV,mass_max_GeV,pt_min_GeV,pt_max_GeV
0,3000,2.60059,3.59878,0.263775,29.71421


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(cms_data["dimuon_mass_GeV"], bins=60, histtype="step", lw=1.8)
axes[0].set_xlabel(r"$m_{\mu\mu}$ (GeV)")
axes[0].set_ylabel("events")
axes[0].set_title("Mass spectrum")

axes[1].hist(cms_data["dimuon_pt_GeV"], bins=50, histtype="step", lw=1.8)
axes[1].set_xlabel(r"dimuon $p_T$ (GeV)")
axes[1].set_ylabel("events")
axes[1].set_title(r"Dimuon $p_T$")
fig.tight_layout()

In [5]:
MASS_RANGE = (2.6, 3.6)
mass_values = cms_data["dimuon_mass_GeV"].to_numpy(dtype=float)
full_fit_result = fit_mass_model(mass_values, MASS_RANGE)
full_fit_parameters = pd.Series(unpack_parameters(full_fit_result.x))
print("fit success:", full_fit_result.success)
print("negative log-likelihood:", full_fit_result.fun)
full_fit_parameters.to_frame("estimate")

fit success: True
negative log-likelihood: -22746.08866334422


,estimate
signal_yield,1576.871285
background_yield,1423.052759
mass_mean_GeV,3.092347
mass_sigma_GeV,0.034693
background_mass_slope,0.717084


# Part 4: Turning the Notebook Calculation into a Script

The file [`scripts/lecture09/dimuon_pt_bootstrap.py`](../scripts/lecture09/dimuon_pt_bootstrap.py) does three things that make it cluster-friendly:

- accepts command-line arguments such as `--pt-min`, `--pt-max`, and `--n-bootstrap`,
- runs one independent pT-bin calculation per process,
- writes one CSV file for that bin.

Run a tiny smoke test before submitting a larger job. This command uses only two bootstrap refits, so it should finish quickly.

In [6]:
SMOKE_TEST_OUTPUT_DIR = Path("../scratch/lecture09") if Path("../scripts/lecture09/").exists() else Path("scratch/lecture09")
SMOKE_TEST_SCRIPT = SCRIPT_DIR / "dimuon_pt_bootstrap.py"
print(f"Running smoke test with script: {SMOKE_TEST_SCRIPT} and data: {DATA_PATH}")

!python -u "{SMOKE_TEST_SCRIPT}" --data "{DATA_PATH}" --pt-min 0 --pt-max 6 --n-bootstrap 2 --output-dir "{SMOKE_TEST_OUTPUT_DIR}" --seed 1

Running smoke test with script: ../scripts/lecture09/dimuon_pt_bootstrap.py and data: ../data/cms_dimuon_jpsi_3000.csv
[2026-09-24 10:17:40] starting dimuon pT-bin bootstrap job
[2026-09-24 10:17:40] input data: ../data/cms_dimuon_jpsi_3000.csv
[2026-09-24 10:17:40] output directory: ../scratch/lecture09
[2026-09-24 10:17:40] pT selection: 0 <= dimuon_pt_GeV < 6 GeV
[2026-09-24 10:17:40] mass selection: 2.6 <= dimuon_mass_GeV <= 3.6 GeV
[2026-09-24 10:17:40] bootstrap resamples: 2
[2026-09-24 10:17:40] random seed: 1
[2026-09-24 10:17:40] loaded 3000 rows in 0.01 s
[2026-09-24 10:17:40] selected 292 rows in 0.00 s
[2026-09-24 10:17:40] selected ranges: mass=[2.60226, 3.59121] GeV, pT=[0.26378, 5.99561] GeV
[2026-09-24 10:17:40] starting original fit
[2026-09-24 10:17:40] original fit finished: success=True, nll=-1403.65, iterations=20, time=0.02 s
[2026-09-24 10:17:40] original fit parameters: signal_yield=38.133, background_yield=253.866, mass_mean=3.073567 GeV, mass_sigma=0.034886 Ge

In [7]:
test_csv = SMOKE_TEST_OUTPUT_DIR / "dimuon_bootstrap_pt_0_6.csv"
pd.read_csv(test_csv).head()

,sample_index,is_original_fit,fit_success,negative_log_likelihood,pt_min_GeV,pt_max_GeV,n_events,signal_yield,background_yield,mass_mean_GeV,mass_sigma_GeV,background_mass_slope
0,-1,True,True,-1403.65026,0.0,6.0,292,38.133136,253.866196,3.073567,0.034886,1.614166
1,0,False,True,-1404.78892,0.0,6.0,292,40.006833,251.992602,3.072846,0.041444,1.703518
2,1,False,True,-1416.82918,0.0,6.0,292,44.155438,247.843278,3.086608,0.029270,1.871894


### Discussion Question

Why is `--n-bootstrap 2` a useful test but not a useful uncertainty estimate? What failure modes can this quick run catch before a Slurm job spends real cluster time?

# Part 5: Submitting pT-Bin Jobs with Slurm

The W&M-specific details live in [`resources/wmhpc.md`](../resources/wmhpc.md). The commands below are the practical minimum for today's workflow.  One that is particularly important is the terminal command to setup your environment to utilize the Slurm commands below

```bash
source /usr/local/etc/sciclone.bashrc 
``` 

or equivalent for `tcsh`/`csh` shell

```csh
source /usr/local/etc/sciclone.cshrc 
``` 

A Slurm batch script is a shell script with resource requests at the top. This repository includes one for today at [`scripts/lecture09/run_dimuon_pt_bin.slurm`](../scripts/lecture09/run_dimuon_pt_bin.slurm). It assumes you submit the job from the repository root after creating and activating the `.venv` once during setup. Each completed job writes a bootstrap CSV and a diagnostic mass-fit PNG into `slurm_outputs/`.

```bash
#!/bin/bash
#SBATCH --job-name=dimuon_pt
#SBATCH --output=slurm_logs/dimuon_pt_%j.out
#SBATCH --error=slurm_logs/dimuon_pt_%j.err
#SBATCH --time=00:20:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G

set -euo pipefail

PT_MIN=$1
PT_MAX=$2
LABEL=$3

cd "${SLURM_SUBMIT_DIR}"

source /usr/local/etc/sciclone.bashrc
source .venv/bin/activate
mkdir -p slurm_outputs

python -u scripts/lecture09/dimuon_pt_bootstrap.py \
  --data data/cms_dimuon_jpsi_3000.csv \
  --pt-min "${PT_MIN}" \
  --pt-max "${PT_MAX}" \
  --label "${LABEL}" \
  --n-bootstrap 200 \
  --output-dir slurm_outputs \
  --seed "${SLURM_JOB_ID}"
```

Before submitting, make sure the log directory exists. Slurm opens the log file before the script body runs.

```bash
mkdir -p slurm_logs
```

Then submit a few independent bins manually from the repository root:

```bash
sbatch scripts/lecture09/run_dimuon_pt_bin.slurm 0 4 pt_0_4
sbatch scripts/lecture09/run_dimuon_pt_bin.slurm 4 8 pt_4_8
sbatch scripts/lecture09/run_dimuon_pt_bin.slurm 8 12 pt_8_12
sbatch scripts/lecture09/run_dimuon_pt_bin.slurm 12 100 pt_12_100
```

For a larger set of bins, you can use a Slurm array, but manual `sbatch` commands are easier to debug the first time.

## Checking Job Status

Useful Slurm commands:

```bash
squeue -u $USER
squeue -j JOBID
scancel JOBID
ls -lh slurm_outputs
```

Common states include `PD` for pending, `R` for running, and `CG` for completing. When jobs finish, inspect both the `.out` and `.err` files. A missing CSV usually means the Python script failed, the path was wrong, the environment was not activated, or the job ran out of time or memory.

# Part 6: Plotting Completed Slurm Outputs

After your jobs finish, point `STUDENT_RESULTS_DIR` to the directory containing your completed CSV files. This may be a cluster path if you are running the notebook on the cluster, or a local path if you copied the CSV files back to your laptop.

In [8]:
# TODO: edit this path to your own completed job directory.
STUDENT_RESULTS_DIR = Path("../slurm_outputs")

result_files = sorted(STUDENT_RESULTS_DIR.glob("dimuon_bootstrap_*.csv"))
print(f"found {len(result_files)} result files")
result_files[:5]

found 4 result files


[PosixPath('../slurm_outputs/dimuon_bootstrap_pt_0_4.csv'),
 PosixPath('../slurm_outputs/dimuon_bootstrap_pt_12_100.csv'),
 PosixPath('../slurm_outputs/dimuon_bootstrap_pt_4_8.csv'),
 PosixPath('../slurm_outputs/dimuon_bootstrap_pt_8_12.csv')]

In [9]:
if not result_files:
    raise FileNotFoundError(
        f"No dimuon_bootstrap_*.csv files found in {STUDENT_RESULTS_DIR}. "
        "Edit STUDENT_RESULTS_DIR after your Slurm jobs finish."
    )

all_results = pd.concat([pd.read_csv(path).assign(source_file=path.name) for path in result_files], ignore_index=True)
all_results.head()

,sample_index,is_original_fit,fit_success,negative_log_likelihood,pt_min_GeV,pt_max_GeV,n_events,signal_yield,background_yield,mass_mean_GeV,mass_sigma_GeV,background_mass_slope,source_file
0,-1,True,True,-353.900284,0.0,4.0,96,5.100725,90.899257,3.059975,0.010000,1.623476,dimuon_bootstrap_pt_0_4.csv
1,0,False,True,-349.012716,0.0,4.0,96,4.737480,91.262031,3.054875,0.010000,1.116877,dimuon_bootstrap_pt_0_4.csv
2,1,False,True,-355.356087,0.0,4.0,96,4.276801,91.723851,3.060588,0.010000,1.806318,dimuon_bootstrap_pt_0_4.csv
3,2,False,True,-350.852252,0.0,4.0,96,9.517466,86.482909,3.057545,0.012967,0.872546,dimuon_bootstrap_pt_0_4.csv
4,3,False,True,-351.994915,0.0,4.0,96,5.232428,90.761909,3.050010,0.010000,1.395827,dimuon_bootstrap_pt_0_4.csv


In [10]:
successful_bootstrap = all_results[(~all_results["is_original_fit"]) & (all_results["fit_success"])]
original_fits = all_results[all_results["is_original_fit"] & all_results["fit_success"]].copy()

summary_rows = []
for (pt_min, pt_max), group in successful_bootstrap.groupby(["pt_min_GeV", "pt_max_GeV"]):
    original = original_fits[(original_fits["pt_min_GeV"] == pt_min) & (original_fits["pt_max_GeV"] == pt_max)].iloc[0]
    summary_rows.append(
        {
            "pt_min_GeV": pt_min,
            "pt_max_GeV": pt_max,
            "pt_center_GeV": 0.5 * (pt_min + pt_max),
            "n_events": int(original["n_events"]),
            "signal_yield_fit": original["signal_yield"],
            "signal_yield_bootstrap_std": group["signal_yield"].std(ddof=1),
            "mass_mean_fit_GeV": original["mass_mean_GeV"],
            "mass_mean_bootstrap_std_GeV": group["mass_mean_GeV"].std(ddof=1),
            "mass_sigma_fit_GeV": original["mass_sigma_GeV"],
            "mass_sigma_bootstrap_std_GeV": group["mass_sigma_GeV"].std(ddof=1),
            "successful_bootstraps": len(group),
        }
    )

pt_fit_summary = pd.DataFrame(summary_rows).sort_values("pt_min_GeV")
pt_fit_summary

,pt_min_GeV,pt_max_GeV,pt_center_GeV,n_events,signal_yield_fit,signal_yield_bootstrap_std,mass_mean_fit_GeV,mass_mean_bootstrap_std_GeV,mass_sigma_fit_GeV,mass_sigma_bootstrap_std_GeV,successful_bootstraps
0,0.0,4.0,2.0,96,5.100725,2.898519,3.059975,0.010528,0.010000,0.003471,200
1,4.0,8.0,6.0,833,408.610579,18.290901,3.090066,0.002404,0.041002,0.002320,200
2,8.0,12.0,10.0,1282,746.019270,22.262323,3.092730,0.001474,0.032664,0.001666,200
3,12.0,100.0,56.0,789,427.628263,17.436947,3.094363,0.001793,0.033136,0.002263,200


In [12]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].errorbar(
    pt_fit_summary["pt_center_GeV"],
    pt_fit_summary["signal_yield_fit"],
    yerr=pt_fit_summary["signal_yield_bootstrap_std"],
    fmt="o",
    capsize=3,
)
axes[0].set_xlabel(r"dimuon $p_T$ bin center (GeV)")
axes[0].set_ylabel("signal yield")

axes[1].errorbar(
    pt_fit_summary["pt_center_GeV"],
    1000.0 * pt_fit_summary["mass_mean_fit_GeV"],
    yerr=1000.0 * pt_fit_summary["mass_mean_bootstrap_std_GeV"],
    fmt="o",
    capsize=3,
)
axes[1].set_xlabel(r"dimuon $p_T$ bin center (GeV)")
axes[1].set_ylabel(r"fitted mass mean (MeV)")

axes[2].errorbar(
    pt_fit_summary["pt_center_GeV"],
    1000.0 * pt_fit_summary["mass_sigma_fit_GeV"],
    yerr=1000.0 * pt_fit_summary["mass_sigma_bootstrap_std_GeV"],
    fmt="o",
    capsize=3,
)
axes[2].set_xlabel(r"dimuon $p_T$ bin center (GeV)")
axes[2].set_ylabel(r"fitted mass width (MeV)")

fig.tight_layout()

### Interpretation Prompts

- Which pT bins have enough events for stable fits?
- Which fitted quantity changes most visibly with pT?
- Do the bootstrap uncertainties grow in bins with fewer selected events?
- If a bin has many failed bootstrap fits, is that a physics result, a modeling problem, or a computing/debugging problem?

# Takeaways

- A notebook is good for exploration, plots, and interpretation. A script is better for repeated batch calculations.
- The smallest useful HPC workflow is: test locally, write a batch script, submit with `sbatch`, monitor with `squeue`, inspect logs, collect outputs.
- Independent pT bins are naturally parallel because each fit reads the same input data but writes a separate output CSV.
- Slurm does not replace debugging. It makes already-debugged repeated work easier to scale.
- The course reference for cluster access and Slurm commands is [`resources/wmhpc.md`](../resources/wmhpc.md).